# 01 — Factor stability diagnostics

> Esta revisión describe qué cambia en la atribución factorial de XLE frente a QQQ y XLF. Es un diagnóstico previo a estudiar modelos de régimen; no introduce una nueva señal ni un backtest.

## <b>Frozen attribution reproduction</b>

Se reproduce la especificación de Block 05: $R_t$ se atribuye con una OLS de 252 sesiones que termina en $t-1$. Los factores contemporáneos se conocen al cierre de $t$, por lo que esto es atribución fuera de muestra del retorno realizado, no predicción previa a la sesión.

In [ ]:
from datetime import date
from pathlib import Path
import math

import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import HTML, display

TARGETS = ('QQQ', 'XLE', 'XLF')
SYMBOLS = ('SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF')
FACTORS = ('market', 'size', 'growth_value')
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)
ROLLING_WINDOW = 252
PERIODS = (('2019–2022', date(2019, 1, 1), date(2022, 12, 31)),
           ('2023–2026', date(2023, 1, 1), RESEARCH_END))

def project_root(start):
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies'
        if (direct / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return direct
        if (parent / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return parent
    raise FileNotFoundError('No se encontró data/gold/daily_series/etf.')

PROJECT_ROOT = project_root(Path.cwd())
GOLD = PROJECT_ROOT / 'data' / 'gold' / 'daily_series' / 'etf'
frames = {
    symbol: pl.read_parquet(GOLD / f'{symbol}.parquet')
    .with_columns(pl.col('session_date').cast(pl.Date))
    .filter(pl.col('session_date').is_between(RESEARCH_START, RESEARCH_END, closed='both'))
    .sort('session_date')
    for symbol in SYMBOLS
}
prices = frames['SPY'].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias('SPY'))
for symbol in SYMBOLS[1:]:
    prices = prices.join(
        frames[symbol].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias(symbol)),
        on='date', how='inner',
    )
returns = prices.sort('date').with_columns([pl.col(symbol).log().diff().alias(f'{symbol}_ret') for symbol in SYMBOLS]).drop_nulls()
model_panel = returns.select('date', *[f'{asset}_ret' for asset in TARGETS], 'SPY_ret', 'IWM_ret', 'IWF_ret', 'IWD_ret').with_columns(
    pl.col('SPY_ret').alias('market'),
    (pl.col('IWM_ret') - pl.col('SPY_ret')).alias('size'),
    (pl.col('IWF_ret') - pl.col('IWD_ret')).alias('growth_value'),
).select('date', *[f'{asset}_ret' for asset in TARGETS], *FACTORS).drop_nulls()

dates = model_panel.get_column('date').to_list()
factor_values = model_panel.select(*FACTORS).to_numpy()
oos_rows = []
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        systematic = beta[0] + factor_values[index] @ beta[1:]
        oos_rows.append({
            'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
            'alpha': beta[0], 'beta_market': beta[1], 'beta_size': beta[2],
            'beta_growth_value': beta[3], 'actual_return': actual[index],
            'systematic_return': systematic, 'residual_return': actual[index] - systematic,
        })
oos = pl.DataFrame(oos_rows)
market_rows = []
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index, :1]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        market_rows.append({'asset': asset, 'actual_return': actual[index],
                            'residual_return': actual[index] - (beta[0] + factor_values[index, 0] * beta[1])})
market_oos = pl.DataFrame(market_rows)

expected_r2 = {'QQQ': 0.972, 'XLE': 0.559, 'XLF': 0.865}
expected_market_r2 = {'QQQ': 0.880, 'XLE': 0.362, 'XLF': 0.705}
reproduction_rows = []
for asset in TARGETS:
    sample = oos.filter(pl.col('asset') == asset)
    actual = sample.get_column('actual_return').to_numpy()
    residual = sample.get_column('residual_return').to_numpy()
    r2 = 1 - (residual ** 2).sum() / ((actual - actual.mean()) ** 2).sum()
    market_sample = market_oos.filter(pl.col('asset') == asset)
    market_actual = market_sample.get_column('actual_return').to_numpy()
    market_residual = market_sample.get_column('residual_return').to_numpy()
    market_r2 = 1 - (market_residual ** 2).sum() / ((market_actual - market_actual.mean()) ** 2).sum()
    reproduction_rows.append({'asset': asset, 'observations': sample.height, 'market_oos_r2': market_r2,
                             'reference_market_r2': expected_market_r2[asset], 'full_oos_r2': r2,
                             'reference_full_r2': expected_r2[asset],
                             'max_abs_difference': max(abs(r2 - expected_r2[asset]), abs(market_r2 - expected_market_r2[asset]))})
reproduction_audit = pl.DataFrame(reproduction_rows).with_columns(pl.all().exclude('asset', 'observations').round(6))
if reproduction_audit.get_column('max_abs_difference').max() > 0.0005:
    raise AssertionError('La reproducción no coincide con el redondeo de Block 05.')
reproduction_audit

## <b>Beta stability</b>

> Las trayectorias rolling son descriptivas y están solapadas. La prueba de interacciones usa retornos diarios y una covarianza HAC de 21 sesiones; no interpreta cada beta rolling como una observación independiente.

In [ ]:
beta_columns = (('beta_market', 'MKT'), ('beta_size', 'SIZE'), ('beta_growth_value', 'GV'))
beta_rows = []
for asset in TARGETS:
    for period, start, end in PERIODS:
        sample = oos.filter((pl.col('asset') == asset) & pl.col('date').is_between(start, end, closed='both'))
        for column, label in beta_columns:
            values = sample.get_column(column).to_numpy()
            beta_rows.append({
                'asset': asset, 'period': period, 'factor': label,
                'median_beta': np.median(values),
                'iqr_beta': np.quantile(values, .75) - np.quantile(values, .25),
                'sign_changes': int(np.sum(np.sign(values[1:]) != np.sign(values[:-1]))),
            })
beta_summary = pl.DataFrame(beta_rows).with_columns(pl.all().exclude('asset', 'period', 'factor', 'sign_changes').round(3))
beta_summary

xle = oos.filter(pl.col('asset') == 'XLE').sort('date')
figure = go.Figure()
for column, label in beta_columns:
    figure.add_scatter(x=xle['date'], y=xle[column], mode='lines', name=label)
figure.add_hline(y=0, line_dash='dash', line_color='gray')
figure.update_layout(title='XLE — rolling factor exposures', xaxis_title='return date', yaxis_title='beta', template='plotly_white')
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))

In [ ]:
def interaction_wald(sample):
    factors = sample.select('market', 'size', 'growth_value').to_numpy()
    later = (sample.get_column('date').to_numpy() >= np.datetime64('2023-01-01')).astype(float)
    design = np.column_stack((np.ones(len(sample)), factors, later, factors * later[:, None]))
    target = sample.get_column('actual_return').to_numpy()
    coefficients = np.linalg.lstsq(design, target, rcond=None)[0]
    residual = target - design @ coefficients
    inverse = np.linalg.pinv(design.T @ design)
    scores = design * residual[:, None]
    meat = scores.T @ scores
    for lag in range(1, 22):
        weight = 1 - lag / 22
        gamma = scores[lag:].T @ scores[:-lag]
        meat += weight * (gamma + gamma.T)
    covariance = inverse @ meat @ inverse
    delta = coefficients[-3:]
    wald = float(delta @ np.linalg.pinv(covariance[-3:, -3:]) @ delta)
    p_value = math.erfc(math.sqrt(wald / 2)) + math.sqrt(2 * wald / math.pi) * math.exp(-wald / 2)
    return delta, wald, p_value

wald_rows = []
for asset in TARGETS:
    sample = oos.filter(pl.col('asset') == asset).join(
        model_panel.select('date', *FACTORS), on='date', how='inner'
    ).sort('date')
    delta, statistic, p_value = interaction_wald(sample)
    wald_rows.append({'asset': asset, 'delta_MKT': delta[0], 'delta_SIZE': delta[1], 'delta_GV': delta[2],
                      'wald_chi2_df3': statistic, 'hac_p_value': f'{p_value:.2e}'})
wald_summary = pl.DataFrame(wald_rows).with_columns(pl.col('delta_MKT', 'delta_SIZE', 'delta_GV', 'wald_chi2_df3').round(3))
wald_summary

## <b>Residual variation</b>

El error de atribución es $e_t = R_t - \hat{R}_{systematic,t}$. Puede contener componente idiosincrático, betas desactualizadas y factores omitidos; no se interpreta automáticamente como ruido irreducible.

In [ ]:
variation_rows = []
for asset in TARGETS:
    for period, start, end in PERIODS:
        sample = oos.filter((pl.col('asset') == asset) & pl.col('date').is_between(start, end, closed='both'))
        actual = sample.get_column('actual_return').to_numpy()
        systematic = sample.get_column('systematic_return').to_numpy()
        residual = sample.get_column('residual_return').to_numpy()
        variation_rows.append({
            'asset': asset, 'period': period, 'n': sample.height,
            'error_daily_vol_pct': residual.std(ddof=1) * 100,
            'mae_pct': np.abs(residual).mean() * 100, 'rmse_pct': np.sqrt(np.mean(residual ** 2)) * 100,
            'var_systematic': np.var(systematic, ddof=1), 'var_error': np.var(residual, ddof=1),
            'cov_systematic_error': np.cov(systematic, residual, ddof=1)[0, 1],
            'var_total': np.var(actual, ddof=1),
            'oos_r2': 1 - (residual ** 2).sum() / ((actual - actual.mean()) ** 2).sum(),
        })
variation_summary = pl.DataFrame(variation_rows).with_columns(
    pl.col('error_daily_vol_pct', 'mae_pct', 'rmse_pct', 'oos_r2').round(3),
    pl.col('var_systematic', 'var_error', 'cov_systematic_error', 'var_total').round(8),
)
variation_summary

rolling_rows = []
for asset in TARGETS:
    sample = oos.filter(pl.col('asset') == asset).sort('date')
    residual = sample.get_column('residual_return').to_numpy()
    for index in range(62, len(sample)):
        rolling_rows.append({'date': sample.get_column('date')[index], 'asset': asset,
                             'rolling_error_vol_pct': residual[index - 62:index + 1].std(ddof=1) * 100})
rolling_error_vol = pl.DataFrame(rolling_rows)
figure = go.Figure()
for asset in TARGETS:
    sample = rolling_error_vol.filter(pl.col('asset') == asset)
    figure.add_scatter(x=sample['date'], y=sample['rolling_error_vol_pct'], mode='lines', name=asset)
figure.update_layout(title='63-session rolling attribution-error volatility', xaxis_title='return date', yaxis_title='daily volatility (%)', template='plotly_white')
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))

In [ ]:
rng = np.random.default_rng(126)
def circular_block_std(values, block=21):
    index = []
    while len(index) < len(values):
        start = rng.integers(0, len(values))
        index.extend((start + np.arange(block)) % len(values))
    return values[np.asarray(index[:len(values)])].std(ddof=1)

bootstrap_rows = []
for asset in TARGETS:
    errors = []
    for _, start, end in PERIODS:
        errors.append(oos.filter((pl.col('asset') == asset) & pl.col('date').is_between(start, end, closed='both')).get_column('residual_return').to_numpy())
    ratio = errors[1].std(ddof=1) / errors[0].std(ddof=1)
    draws = np.asarray([circular_block_std(errors[1]) / circular_block_std(errors[0]) for _ in range(2_000)])
    bootstrap_rows.append({'asset': asset, 'vol_ratio_2023_2026_to_2019_2022': ratio,
                           'bootstrap_p05': np.quantile(draws, .05), 'bootstrap_median': np.quantile(draws, .50),
                           'bootstrap_p95': np.quantile(draws, .95), 'probability_ratio_gt_1': (draws > 1).mean()})
bootstrap_summary = pl.DataFrame(bootstrap_rows).with_columns(pl.all().exclude('asset').round(3))
bootstrap_summary

## <b>Interpretation</b>

- La tabla de reproducción debe coincidir con los R² OOS redondeados de Block 05 antes de interpretar los diagnósticos.
- Un Wald-HAC pequeño no prueba estabilidad; sólo indica que esta especificación y este corte no separan con precisión las pendientes. Un valor grande es evidencia descriptiva de cambio, no confirmatoria.
- La variación de error, su covarianza con la parte sistemática y la caída de R² distinguen un problema de cobertura de factores de un simple aumento de volatilidad.
- El bootstrap cuantifica incertidumbre dependiente en el tiempo para un único corte y longitud de bloque fijados. No corrige la exploración previa de las mismas fechas.
- Betas inestables junto con mayor error residual justificarían evaluar modelos de betas dinámicas o cambios estructurales. No implican por sí solos que XLE Systematic Momentum deje de ser rentable.